In [2]:
import cv2
import numpy as np 
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
import os
from skimage.measure import shannon_entropy
from skimage.feature import local_binary_pattern

#local binary pattern of image
def loc_bin_mean(img):
    radius = 1
    n_points = 8*radius

    lbp = local_binary_pattern(img , n_points,radius,method ='uniform')
    return np.mean(lbp)

#fft energy of feature 
def compute_fft(image):
    f = np.fft.fft2(image)             
    fshift = np.fft.fftshift(f)        
    magnitude = np.log(1 + np.abs(fshift))  
    energy = np.sum(magnitude ** 2)
    return energy

#std dev of noise in image
def get_noise(img):
    blur = cv2.GaussianBlur(img, (5,5), 0)
    noise = img - blur
    std = np.std(noise)
    return std
#entropy of image 
def get_entropy(img):
    return shannon_entropy(img)

#edge density and strength
def edge_analysis(img):
    edges = cv2.Canny(img, 150, 250)
    edge_density = np.sum(edges > 0) / edges.size
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    return edge_density,np.mean(magnitude)

#eggb variance of image
def get_rgb_variance(img):
    r,g,b = cv2.split(img)

    var = np.var(r) + np.var(g) + np.var(b)
    return var

def glcm_features(img):
    # Convert to 8-bit (important)
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
    
    # Compute GLCM
    glcm = graycomatrix(img, distances=[1], angles=[0], 
                        levels=256, symmetric=True, normed=True)
    
    # Extract features
    contrast = graycoprops(glcm, 'contrast')[0,0] # Real -> high , Ai -> low 
    homogeneity = graycoprops(glcm, 'homogeneity')[0,0]# Real -> low , ai - High
    
    return contrast, homogeneity

def laplacian_analysis(img):
    # Apply Laplacian
    lap = cv2.Laplacian(img, cv2.CV_64F)
    
    # Compute variance (key feature)
    lap_var = np.var(lap)
    
    return lap_var

#extracct the impo features from the images
def features_extraction(image):
    img = cv2.imread(image)
    if img is None:
        return None
    color_img = img.copy()
    img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

    #local binary pattern
    lbp = loc_bin_mean(img)

    #fft energy
    fft_energy = compute_fft(img)

    #entropy of image
    entropy = get_entropy(img)

    #rgb variance of image
    rgb_variance = get_rgb_variance(color_img)

    #noise of the image
    noise_std = get_noise(img)

    #edge density and strength of the image
    edge_density,edge_strength = edge_analysis(img)

    #contrast and homogeneity of the image
    contrast, homogeneity = glcm_features(img)

    #laplacian variance of the image
    laplacian_variance = laplacian_analysis(img)

    return [lbp, fft_energy, entropy, rgb_variance, noise_std, edge_density, edge_strength, contrast, homogeneity, laplacian_variance]


image_dataset = []
real_image_folder = 'RealArt'
ai_image_folder='AiArtData'


for file in os.listdir(real_image_folder):
    path = os.path.join(real_image_folder,file)

    features = features_extraction(path)

    if features is not None:
        features.append("Real")
        image_dataset.append(features)


for file in os.listdir(ai_image_folder):
    path = os.path.join(ai_image_folder,file)

    features = features_extraction(path)

    if features is not None:
        features.append("AI")
        image_dataset.append(features)

columns = ['lbp', 'fft_energy', 'entropy', 'rgb_variance', 'noise_std', 'edge_density', 'edge_strength', 'contrast', 'homogeneity', 'laplacian_variance','label']

df = pd.DataFrame(image_dataset,columns = columns)

df = df.sample(frac= 1,random_state=42).reset_index(drop=True)

df.to_csv("Image_features_Dataset.csv",index = False)

print("Dataset is created!!!!")
   

Dataset is created!!!!


In [3]:
df.head()

,lbp,fft_energy,entropy,rgb_variance,noise_std,edge_density,edge_strength,contrast,homogeneity,laplacian_variance,label
0,5.065186,2.135198e+08,7.072803,4105.900556,109.640449,0.027275,33.473377,167.815542,0.517794,1088.392427,Real
1,4.729126,3.238841e+06,7.930742,14424.042733,117.817730,0.155832,123.037011,693.892917,0.141047,3000.091484,AI
2,5.628545,4.143853e+08,7.009984,17415.115243,109.600934,0.002737,14.320840,16.390790,0.576594,35.775127,Real
3,5.559313,2.918481e+07,7.766271,12704.539557,102.610988,0.019975,24.271669,64.730712,0.552029,238.014602,AI
4,5.043193,1.250291e+08,7.499648,10006.370715,119.735619,0.041766,48.827804,143.482714,0.344421,832.417998,Real


In [4]:
df.shape

(975, 11)

In [5]:
print("Real images:", len(os.listdir(real_image_folder)))
print("AI images:", len(os.listdir(ai_image_folder)))

Real images: 436
AI images: 539
